# Textual Inversion Demo

This notebook demonstrates how to use the refactored `textual_inversion` package to fine-tune a Stable Diffusion model using Keras.

In [ ]:
# 1. Install dependencies (uncomment if running in Colab)
# !pip install -q keras_cv==0.4.0
# !pip install -q -U tensorflow
# !pip install -q -U protobuf

In [ ]:
import tensorflow as tf
from tensorflow import keras
import keras_cv

from textual_inversion.config import TrainingConfig
from textual_inversion.data.dataset import assemble_dataset
from textual_inversion.models.trainer import TextualInversionTrainer
from textual_inversion.callbacks.image_generator import ImageGenerationCallback
from textual_inversion.utils.visualization import plot_images

## Configuration

In [ ]:
config = TrainingConfig(
    placeholder_token="espositope",
    subject="cat",
    epochs=50
)

## Load Base Model

In [ ]:
stable_diffusion = keras_cv.models.StableDiffusion()
stable_diffusion.tokenizer.add_tokens(config.placeholder_token)

noise_scheduler = keras_cv.models.stable_diffusion.NoiseScheduler(
    beta_start=config.beta_start,
    beta_end=config.beta_end,
    beta_schedule=config.beta_schedule,
    train_timesteps=config.train_timesteps,
)

## Setup Dataset
Provide a list of absolute paths to your training images.

In [ ]:
file_paths = [
    # "/path/to/your/cat1.jpg",
]

if file_paths:
    train_ds = assemble_dataset(
        file_paths=file_paths,
        config=config,
        tokenizer=stable_diffusion.tokenizer
    )
    train_ds = train_ds.batch(config.batch_size).shuffle(
        train_ds.cardinality(), reshuffle_each_iteration=True
    )

## Initialize Trainer

In [ ]:
trainer = TextualInversionTrainer(
    stable_diffusion=stable_diffusion,
    noise_scheduler=noise_scheduler,
    config=config,
    name="trainer"
)

if file_paths:
    learning_rate = keras.optimizers.schedules.CosineDecay(
        initial_learning_rate=config.initial_learning_rate, 
        decay_steps=train_ds.cardinality() * config.epochs
    )
    
    optimizer = keras.optimizers.Adam(
        weight_decay=config.weight_decay, 
        learning_rate=learning_rate, 
        epsilon=config.epsilon, 
        global_clipnorm=config.global_clipnorm
    )

    trainer.compile(
        optimizer=optimizer,
        loss=keras.losses.MeanSquaredError(reduction="none"),
    )

## Train

In [ ]:
cbs = [
    ImageGenerationCallback(
        stable_diffusion=stable_diffusion,
        prompt=f"an oil painting of {config.placeholder_token}",
        seed=config.seed
    )
]

if file_paths:
    trainer.fit(
        train_ds,
        epochs=config.epochs,
        callbacks=cbs,
    )